# Resource Demand Predictive

## Problem Framing

**Business question:** How many active residents are likely to need support at each safehouse next month?

**Primary users:** operations leadership, fundraising leadership

**Decision supported:** Support staffing and fundraising planning with a forecast of near-term resident demand by site.

**Modeling goal:** Prediction. This notebook is judged first on how well it scores new, unseen records rather than on whether a single coefficient is easy to interpret.

**Target / outcome anchor:** Current regression target: `label_next_active_residents`, forecasting next-month resident load from safehouse-month patterns.

**Submission status:** Artifact-backed predictive pipeline. The repository includes committed model and evaluation artifacts for this track.

Prediction is the right framing here because the organization needs an operational ranking or forecast that can be applied to future records. This notebook therefore prioritizes leakage control, reproducible feature availability at scoring time, and honest out-of-sample validation. Causal interpretation is handled more carefully in the paired explanatory notebook for the same pipeline.


## Data Acquisition, Preparation & Exploration

**Data source and grain:** This notebook loads `safehouse_monthly_features`, which contains safehouse-month snapshots that combine occupancy, service, staffing, and operating signals at the site-month grain.

**Reproducible preparation pipeline:** The code cells below use the shared notebook bootstrap plus `load_notebook_context(...)` so the data loading, joins, cleaning, and feature engineering come from reusable modules under `ml/src/data/`, `ml/src/features/`, and the pipeline-specific implementation for `resource_demand` rather than from one-off notebook code.

**Exploration focus:** Before trusting model output, review the shared table preview, target availability, missingness patterns, date coverage, outliers, and any fields that appear to encode post-outcome information. The goal is to confirm that the data can answer the business question without leakage.

**Shared references used by this notebook:**
* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-b-notebook-standardization.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`


In [1]:
print("Notebook execution proof: resource_demand predictive template rendered successfully.")


Notebook execution proof: resource_demand predictive template rendered successfully.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='resource_demand',
    dataset_name='safehouse_monthly_features',
    predictive_impl='resource_demand',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


## Modeling & Feature Selection

This notebook is prediction-first, so candidate models should be compared using the saved evaluation bundle and the strongest out-of-sample metric for this target. The preferred model is whichever generalizes best while still being stable enough to deploy in staff workflows.

Feature selection follows four rules in this submission:

1. Keep only fields available at scoring time.
2. Remove identifiers, direct future labels, and post-outcome leakage.
3. Favor features with stable business meaning and tolerable missingness.
4. Use domain reasoning alongside model importance so the final feature set is defensible.

**Committed artifact summary:** The committed best model is `random_forest_regressor` for target `label_next_active_residents`, selected on `r2` = `1.0000`, using `324` training rows and `117` holdout rows.

**Feature inventory:** 111 engineered inputs are currently stored in `ml/models/resource_demand/feature_list.json`.


## Evaluation & Interpretation

**Validation discipline:** The code cells below pull the saved metrics JSON, holdout comparison table, and cross-validation summary for this pipeline. Those artifacts are the correct basis for judging model quality, not in-sample fit.

**Business meaning of the score:** A high score means the record looks more likely to match this pipeline's target outcome on unseen future data, so `Support staffing and fundraising planning with a forecast of near-term resident demand by site.` can be prioritized earlier.

**Real-world error consequences:** False negatives matter because they can hide girls, cases, or sites that need early intervention. False positives matter because staff time, bed capacity, and follow-up slots are limited and can be pulled away from higher-need situations.

Operationally, the right threshold depends on the workflow. Teams should read precision, recall, ranking quality, and class balance together before deciding whether the score should drive alerts, queue ordering, or a softer recommendation panel.


In [ ]:
evaluation = load_evaluation_bundle('resource_demand')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


In [ ]:
summarize_frame(holdout_comparison)


In [ ]:
summarize_frame(cv_summary)


## Causal and Relationship Analysis

This notebook uses predictive relationships, not causal identification. Strong features matter because they help separate future outcomes on unseen data, but they do **not** prove that changing the feature will change the outcome.

**Most impactful committed features:** The strongest committed model signals currently surfaced in the repo are `active residents`, `capacity utilization ratio`, `capacity girls`, `capacity gap`, `city General Santos`. These are useful for prioritization and investigation, but they still represent association rather than proof of causation.

**Decision guidance:** Use these high-signal features to focus staff review, choose follow-up questions, and prioritize intervention or outreach. When leadership wants to argue that a driver should become policy, treatment, or strategy, they should pair this notebook with the explanatory companion and with domain judgment about omitted variables, timing, and selection bias.


## Deployment Notes

**Canonical widgets for the web app:**
* `forecast_widget`
* `insight_summary_card`
* `ranked_table_widget`

**Submission deployment notes:**
* Use a forecast widget and ranked planning table in monthly operations reviews.
* Pair the forecast with recent occupancy context so leaders can see whether the demand signal reflects a sustained load or a temporary shift.

**Artifact locations referenced by this notebook:**
* `ml/models/resource_demand/metrics.json`
* `ml/models/resource_demand/feature_list.json`
* `ml/models/resource_demand/explainability.csv`
* `ml/reports/evaluation/resource_demand_metrics.json`

**Endpoint pattern used in the app layer when this track is scored:**
* `POST /ml/predict/resource_demand`
* `POST /ml/score-batch/resource_demand`
